# ProductIQ Phase 3 — Step 3.4: Product Text Representation

Deterministic product text from `ProductRepresentation` (structured attributes only; no LLM).

In [2]:
from __future__ import annotations

from pathlib import Path

import pandas as pd

from productiq.representation import build_product_representation, build_product_text
from productiq.representation.text import render_machine_token

# Phase 2.9 artifact name (same value as data.export.schema.PROCESSED_PARQUET_FILENAME).
PROCESSED_PARQUET_FILENAME = "product_catalog.parquet"


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    msg = "Could not locate ProductIQ project root"
    raise RuntimeError(msg)


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
PARQUET_PATH = PROJECT_ROOT / "resources" / "processed" / PROCESSED_PARQUET_FILENAME
SAMPLE_SIZE = 200

catalog = pd.read_parquet(PARQUET_PATH, engine="pyarrow").head(SAMPLE_SIZE)
representations = [
    build_product_representation(row)
    for row in catalog.to_dict(orient="records")
]
print("Representations:", len(representations))

Representations: 200


## 1. Generate product text

In [3]:
texts = [build_product_text(rep) for rep in representations]
for rep, text in zip(representations[:3], texts[:3], strict=True):
    print(rep.product_id)
    print(text)
    print("---")

441137362002
Brand: Netplay. Category: Men. Product Type: T-Shirt. Color: White. Pattern: Checked.
---
441124497006
Brand: Netplay. Category: Men. Product Type: Trousers. Color: Navy. Fit: Tapered Fit. Style: Flat Front.
---
460453612003
Brand: The-Indian-Garage-Co. Category: Men. Product Type: Shirt. Color: White. Pattern: Striped. Fit: Slim Fit. Features: Patch Pocket, Pocket.
---


## 2. Rich attribute coverage

In [4]:
def attribute_count(rep) -> int:
    fields = [
        rep.brand or rep.brand_normalized,
        rep.product_type,
        rep.color,
        rep.pattern,
        rep.material,
        rep.fit,
        rep.sleeve,
        rep.neckline,
        rep.product_features,
        rep.style_attributes,
    ]
    return sum(1 for value in fields if value)


richest = max(representations, key=attribute_count)
print("Richest sample:", attribute_count(richest), "optional fields")
print(build_product_text(richest))

Richest sample: 6 optional fields
Brand: The-Indian-Garage-Co. Category: Men. Product Type: Shirt. Color: White. Pattern: Striped. Fit: Slim Fit. Features: Patch Pocket, Pocket.


## 3. Sparse / missing attributes

In [5]:
sparse = min(representations, key=attribute_count)
print("Sparse sample:", attribute_count(sparse), "optional fields")
print(build_product_text(sparse))

Sparse sample: 2 optional fields
Brand: Dnmx. Category: Men. Color: Maroon.


## 4. Multi-value rendering

In [7]:
def find_multivalue_representation(reps):
    for rep in reps:
        candidates = (
            ("color", rep.color),
            ("pattern", rep.pattern),
            ("material", rep.material),
            ("fit", rep.fit),
            ("sleeve", rep.sleeve),
            ("neckline", rep.neckline),
            ("product_features", rep.product_features),
            ("style_attributes", rep.style_attributes),
        )
        for name, values in candidates:
            if values and len(values) > 1:
                return rep, name, values
    return None, None, None


multi, field_name, field_values = find_multivalue_representation(representations)
if multi is None:
    msg = "No multi-value attributes in sample; increase SAMPLE_SIZE."
    raise RuntimeError(msg)

print(f"multi-value field: {field_name}", field_values)
print(build_product_text(multi))

multi-value field: product_features ['patch_pocket', 'pocket']
Brand: The-Indian-Garage-Co. Category: Men. Product Type: Shirt. Color: White. Pattern: Striped. Fit: Slim Fit. Features: Patch Pocket, Pocket.


## 5. Determinism

In [8]:
sample = representations[0]
a = build_product_text(sample)
b = build_product_text(sample)
assert a == b
print("Deterministic:", a == b)

Deterministic: True


## 6. Readable token rendering

In [9]:
examples = ["navy_blue", "three_quarter", "t_shirt", "chest_pocket"]
for token in examples:
    print(token, "->", render_machine_token(token))

navy_blue -> Navy Blue
three_quarter -> Three Quarter
t_shirt -> T-Shirt
chest_pocket -> Chest Pocket
